In [1]:
!pip install -q rdflib google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 9.9 MB/s eta 0:00:00


In [1]:
import rdflib
import google.generativeai as genai
import os
from google.colab import userdata

# Configuración de la API de Gemini
# Es recomendable usar variables de entorno: os.environ.get("GEMINI_API_KEY")
API_KEY = userdata.get('GOOGLE_API_MTI')
genai.configure(api_key=API_KEY)

# Instanciar el modelo (usaremos Gemini 1.5 Flash por su velocidad en RAG, o Pro para razonamiento complejo)
model = genai.GenerativeModel('gemini-3.1-flash-lite-preview')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [2]:
# Inicializar el grafo y cargar los datos
g = rdflib.Graph()

# Asegúrate de que 'datos.ttl' esté en el mismo directorio que tu Notebook
g.parse("/content/datos.ttl", format="turtle")

print(f"Grafo cargado exitosamente. Contiene {len(g)} tripletas.")

Grafo cargado exitosamente. Contiene 2525 tripletas.


In [3]:
def extraer_contexto_conciertos(nombre_persona):
    # Consulta SPARQL para buscar qué conciertos (P1344) asistió el usuario
    # y recuperar la etiqueta, banda (P175) y país/año (P276).
    query = f"""
    PREFIX : <http://example.org/>
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX p: <https://www.wikidata.org/wiki/Property:>

    SELECT ?persona ?labelConcierto
    WHERE {{
        :{nombre_persona} p:P1344 ?concierto .
        ?concierto :label ?labelConcierto .
    }}
    """

    resultados = g.query(query)

    contexto_extraido = f"Datos extraídos del Grafo de Conocimiento para {nombre_persona}:\n"
    for row in resultados:
        contexto_extraido += f"- Asistió al evento catalogado como: {row.labelConcierto}\n"

    return contexto_extraido

# Probamos la función con el nodo de Mauricio
contexto_grafo = extraer_contexto_conciertos("Mauricio_Beltran")
print("Contexto generado:\n", contexto_grafo)

Contexto generado:
 Datos extraídos del Grafo de Conocimiento para Mauricio_Beltran:
- Asistió al evento catalogado como: Concierto de Dream Theater en Movistar Arena (2008)
- Asistió al evento catalogado como: Concierto de Dream Theater en Movistar Arena (2010)
- Asistió al evento catalogado como: Concierto de Dream Theater en Movistar Arena (2024)
- Asistió al evento catalogado como: Concierto de Symphony X en Teatro Caupolicán (2007)
- Asistió al evento catalogado como: Concierto de Symphony X en Blondie (2019)
- Asistió al evento catalogado como: Concierto de Symphony X en Teatro Caupolicán (2022)
- Asistió al evento catalogado como: Concierto de Symphony X en Teatro Coliseo (2024)
- Asistió al evento catalogado como: Concierto de Symphony X en Teatro Teletón (2026)
- Asistió al evento catalogado como: Concierto de Avantasia en Teatro Caupolicán (2013)
- Asistió al evento catalogado como: Concierto de Avantasia en Teatro Caupolicán (2019)
- Asistió al evento catalogado como: Concie

In [4]:
def consultar_gemini_con_grafo(pregunta_usuario, contexto):
    prompt_template = f"""
    Eres un asistente experto en analizar ontologías y grafos de conocimiento musicales.
    Utiliza estrictamente la siguiente información extraída del grafo para responder a la pregunta del usuario.
    Si la respuesta no está en el contexto, indica que no tienes suficientes datos en el grafo.

    CONTEXTO DEL GRAFO:
    {contexto}

    PREGUNTA DEL USUARIO:
    {pregunta_usuario}

    Respuesta:
    """

    respuesta = model.generate_content(prompt_template)
    return respuesta.text

# Ejecución del pipeline completo
pregunta = "Resume el historial de conciertos en vivo de Mauricio y analiza brevemente qué géneros predominan en sus gustos según los eventos a los que fue."
respuesta_final = consultar_gemini_con_grafo(pregunta, contexto_grafo)

print("### Respuesta del LLM (GraphRAG) ###\n")
print(respuesta_final)

### Respuesta del LLM (GraphRAG) ###

Basado en el historial de conciertos de Mauricio Beltrán, aquí presento el resumen y análisis solicitado:

### Resumen del historial de conciertos
Mauricio ha asistido a un total de 13 eventos musicales registrados entre los años 2007 y 2026, centrados principalmente en tres bandas recurrentes y una aparición puntual:

*   **Dream Theater:** 3 asistencias (2008, 2010, 2024), todas en el Movistar Arena.
*   **Symphony X:** 5 asistencias (2007, 2019, 2022, 2024, 2026) en diversos recintos (Teatro Caupolicán, Blondie, Teatro Coliseo, Teatro Teletón).
*   **Avantasia:** 4 asistencias (2013, 2019, 2023, 2025) principalmente en el Teatro Caupolicán y una en el Teatro Coliseo.
*   **Helloween:** 1 asistencia (2017) en el Movistar Arena.

### Análisis de géneros predominantes
A partir de los artistas mencionados, se observa una clara predominancia del **Metal Progresivo** y el **Power Metal**.

*   **Dream Theater y Symphony X** son exponentes fundamentale